In [7]:
import json
import os
import shutil
from pathlib import Path
from collections import defaultdict
from PIL import Image

train_coco_json = Path('../data/processed/training_coco.json')
test_coco_json = Path('../data/processed/test_coco.json')

DIR_IMAGES = Path('../data/raw/images/')
OUTPUT_DIR = Path('../data/processed/dataset/images/')

In [8]:
def create_dirs():
    train_dir = OUTPUT_DIR / 'train'
    test_dir = OUTPUT_DIR / 'test'
    train_dir.mkdir(parents=True, exist_ok=True)
    test_dir.mkdir(parents=True, exist_ok=True)
    
    return train_dir, test_dir

def copy_images(image_paths, dest_dir):
    
    missing_images = []
    for image_path in image_paths:
        image_name = Path(image_path).name
        src_image = DIR_IMAGES / image_name
        if src_image.exists():
            shutil.copy(src_image, dest_dir)
        else:
            missing_images.append(image_name)
            
    return missing_images

def process_coco_json(coco_json_path):
    with open(coco_json_path, 'r') as f:
        coco_data = json.load(f)
    
    image_paths = []

    for img_info in coco_data['images']:
        image_paths.append(img_info['file_name'])
    
    return image_paths

In [9]:
def validate_images(image_paths, dest_dir):
    missing_images = []
    for image_path in image_paths:
        dest_image = dest_dir / image_path
        if not dest_image.exists():
            missing_images.append(image_path)
    
    return missing_images

In [10]:
def validate_image_integrity(dest_dir):
    invalid_images = []
    for image_path in dest_dir.iterdir():
        if image_path.suffix.lower() not in ['.jpg', '.jpeg', '.png']:
            invalid_images.append(image_path.name)
            continue
        try:
            with Image.open(image_path) as img:
                img.verify()  # Verifies the integrity of the image (does not open the image, only validates)
        except (IOError, SyntaxError) as e:
            invalid_images.append(image_path.name)
    
    return invalid_images

In [11]:
train_dir, test_dir = create_dirs()

train_image_paths = process_coco_json(train_coco_json)
print(f"Copying {len(train_image_paths)} images to the training directory")
missing_train_images = copy_images(train_image_paths, train_dir)
if missing_train_images:
    print(f"Missing training images: {missing_train_images}")

test_image_paths = process_coco_json(test_coco_json)
print(f"Copying {len(test_image_paths)} images to the testing directory")
missing_test_images = copy_images(test_image_paths, test_dir)
if missing_test_images:
    print(f"Missing testing images: {missing_test_images}") 
    
missing_train_images = validate_images(train_image_paths, train_dir)
missing_test_images = validate_images(test_image_paths, test_dir)

if missing_train_images:
    print(f"After copy, missing training images: {missing_train_images}")
if missing_test_images:
    print(f"After copy, missing testing images: {missing_test_images}")
    
invalid_train_images = validate_image_integrity(train_dir)
invalid_test_images = validate_image_integrity(test_dir)

if invalid_train_images:
    print(f"Invalid training images: {invalid_train_images}")
if invalid_test_images: 
    print(f"Invalid testing images: {invalid_test_images}")

Copying 1208 images to the training directory
Copying 120 images to the testing directory


### Compare dir images with JSON COCO

In [12]:
dir_path = '../data/processed/dataset/images/train'

files = [f for f in os.listdir(dir_path) if os.path.isfile(os.path.join(dir_path, f))]

with open(train_coco_json, 'r') as f:
    coco_data = json.load(f)

train_image_paths = [img_info['file_name'] for img_info in coco_data['images']]

missing_files = [file for file in train_image_paths if file not in files]
extra_files = [file for file in files if file not in train_image_paths]

if missing_files:
    print("Files missing in the 'train' directory:")
    for file in missing_files:
        print(f"- {file}")
else:
    print("All files from the JSON are present in the 'train' directory.")
if extra_files:
    print("\nExtra files in the 'train' directory that are not in the JSON:")
    for file in extra_files:
        print(f"- {file}")
else:
    print("There are no extra files in the 'train' directory.")

All files from the JSON are present in the 'train' directory.
There are no extra files in the 'train' directory.


In [13]:
dir_path = '../data/processed/dataset/images/test'

files = [f for f in os.listdir(dir_path) if os.path.isfile(os.path.join(dir_path, f))]

with open(test_coco_json, 'r') as f:
    coco_data = json.load(f)

test_image_paths = [img_info['file_name'] for img_info in coco_data['images']]

missing_files = [file for file in test_image_paths if file not in files]
extra_files = [file for file in files if file not in test_image_paths]

if missing_files:
    print("Files missing in the 'test' directory:")
    for file in missing_files:
        print(f"- {file}")
else:
    print("All files from the JSON are present in the 'test' directory.")
if extra_files:
    print("\nExtra files in the 'test' directory that are not in the JSON:")
    for file in extra_files:
        print(f"- {file}")
else:
    print("There are no extra files in the 'test' directory.")

All files from the JSON are present in the 'test' directory.
There are no extra files in the 'test' directory.
